# ⚖️ Notebook 4 — Model Comparison & Trade-offs

Side-by-side evaluation of Prophet vs LSTM on the same test set.

## 4.1 Setup

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.join('..', 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data_preprocessing import (load_and_clean, aggregate_monthly, train_test_split_ts,
                                 create_lstm_sequences, scale_series)
from prophet_model import train_prophet, forecast_prophet, evaluate_on_test
from lstm_model import build_lstm_model, train_lstm, predict_lstm
from evaluate import evaluate_model, plot_comparison, plot_actual_vs_predicted

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline
LOOKBACK = 12; TEST_MONTHS = 6
os.makedirs('../outputs', exist_ok=True)


## 4.2 Prepare Data (shared between both models)

In [ ]:
df = load_and_clean('../data/train.csv')
monthly = aggregate_monthly(df)
train_df, test_df = train_test_split_ts(monthly, test_months=TEST_MONTHS)
train_vals, test_vals = train_df['y'].values, test_df['y'].values


## 4.3 Run Prophet

In [ ]:
prophet_model = train_prophet(train_df)
forecast = forecast_prophet(prophet_model, periods=TEST_MONTHS)
prophet_preds = evaluate_on_test(forecast, test_df)
prophet_result = evaluate_model('Prophet', test_df['y'].values, prophet_preds)


## 4.4 Run LSTM

In [ ]:
train_scaled, test_scaled, scaler = scale_series(train_vals, test_vals)
full_scaled = np.concatenate([train_scaled, test_scaled])
split_idx = len(train_scaled)

X_all, y_all = create_lstm_sequences(full_scaled, lookback=LOOKBACK)
X_train = X_all[:split_idx-LOOKBACK]; y_train = y_all[:split_idx-LOOKBACK]
X_test  = X_all[split_idx-LOOKBACK:]; y_test  = y_all[split_idx-LOOKBACK:]

lstm_model = build_lstm_model(lookback=LOOKBACK, lstm_units_1=64, lstm_units_2=32)
train_lstm(lstm_model, X_train, y_train, epochs=150, batch_size=4)

lstm_preds   = predict_lstm(lstm_model, X_test, scaler)
test_actual  = scaler.inverse_transform(y_test.reshape(-1,1)).flatten()
lstm_result  = evaluate_model('LSTM', test_actual, lstm_preds)


## 4.5 Metric Comparison Bar Chart

In [ ]:
plot_comparison(
    [prophet_result, lstm_result],
    save_path='../outputs/model_comparison_metrics.png'
)


## 4.6 Overlay: Actual vs Both Models

In [ ]:
common_len = min(len(prophet_preds), len(lstm_preds))
plot_actual_vs_predicted(
    test_df['ds'].values[-common_len:],
    test_df['y'].values[-common_len:],
    {'Prophet': prophet_preds[-common_len:], 'LSTM': lstm_preds[-common_len:]},
    title='Sales Forecast — Prophet vs LSTM vs Actuals',
    save_path='../outputs/model_comparison_overlay.png'
)


## 4.7 Summary Table

In [ ]:
results_df = pd.DataFrame([prophet_result, lstm_result])
results_df['MAE']  = results_df['MAE'].apply(lambda x: f'${x:,.0f}')
results_df['RMSE'] = results_df['RMSE'].apply(lambda x: f'${x:,.0f}')
results_df['MAPE'] = results_df['MAPE'].apply(lambda x: f'{x:.2f}%')
print(results_df.to_string(index=False))
results_df.to_csv('../outputs/results_summary.csv', index=False)


## 4.8 Trade-offs Discussion

### When to use Prophet
- **Business settings** where explainability is required (explain results to non-technical stakeholders)
- **Smaller datasets** (< 2 years of monthly data)
- When you need **fast iteration** — train in seconds, tune with 2–3 params
- When **seasonality and holidays** are known and important

### When to use LSTM
- **Large datasets** with complex, non-linear patterns
- When you have **multivariate features** (weather, promotions, etc.)
- When raw forecast accuracy matters more than interpretability
- When you're comfortable with deep learning infrastructure

### Key Insight
> Prophet's greatest strength is not necessarily accuracy — it's **interpretability**.  
> Its component plots let a business analyst understand *why* sales peak in November  
> without any ML expertise. LSTM treats the process as a black box.

### Accuracy vs Interpretability Spectrum

```
More Interpretable ◄─────────────────────────────► More Accurate (complex data)
   Naive / Moving Avg → Prophet → ARIMA → XGBoost → LSTM → Transformer
```


## 4.9 Recommendations for This Dataset

1. **Use Prophet** for monthly reporting and stakeholder dashboards — it captures Q4 seasonality cleanly
2. **Use LSTM** if you add external features (promotions, holidays, category dummies) and have more data
3. Consider an **ensemble** (average Prophet + LSTM predictions) for production systems
4. Retrain both models quarterly as new data arrives